# 🛡️ TruthRadar: NLP & Machine Learning Experiment Notebook
### AI-Powered Fake News Detection & Explainability System

This notebook provides a complete walkthrough of the experimental pipeline:
1. **Data Ingestion & Preprocessing**
2. **TF-IDF + Logistic Regression Classifier**
3. **Model Evaluation & Confusion Matrix**
4. **DistilBERT Contextual Embeddings**
5. **Headline-Body Semantic Similarity (Sentence-Transformers)**
6. **Sentiment Analysis (NLTK VADER)**
7. **Named Entity Recognition (spaCy)**
8. **Explainable AI (LIME Text Explanations)**
9. **Calibrated Multi-Signal Ensemble Scoring**

In [ ]:
import sys
import os
from pathlib import Path

# Setup root project directory
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.preprocessor import clean_text, preprocess_for_ml
from training.train_tfidf import load_dataset

print("Environment initialized successfully.")
print(f"Project root: {ROOT_DIR}")

## 1. Dataset Loading & Inspection

In [ ]:
# Load sample dataset
data_file = ROOT_DIR / "data" / "sample_news.csv"
df = load_dataset(data_file)

print(f"Dataset Shape: {df.shape}")
display(df.head())

## 2. Exploratory Data Analysis & Class Distribution

In [ ]:
class_counts = df['target'].value_counts()
labels = ['REAL (0)', 'FAKE (1)']

plt.figure(figsize=(6, 4))
sns.barplot(x=labels, y=class_counts.values, palette=['#16A34A', '#DC2626'])
plt.title('Dataset Target Class Distribution', fontsize=12, fontweight='bold')
plt.ylabel('Number of Samples')
plt.xlabel('Class Label')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3. Preprocessing and Feature Extraction (TF-IDF)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Clean text for statistical ML
df['cleaned_text'] = df['combined_text'].apply(preprocess_for_ml)

X = df['cleaned_text'].values
y = df['target'].values

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"TF-IDF Feature Space Dimensions: {X_train_vec.shape}")

## 4. Model Training & Evaluation

In [ ]:
clf = LogisticRegression(C=1.0, random_state=42, class_weight='balanced')
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['REAL', 'FAKE']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred REAL', 'Pred FAKE'], yticklabels=['Actual REAL', 'Actual FAKE'])
plt.title('Confusion Matrix (Test Set)')
plt.show()

## 5. Headline-Body Semantic Consistency Check

In [ ]:
from src.headline_checker import check_headline_body_consistency

headline = "Scientists Discover High-Efficiency Solar Cell"
body = "Researchers at the National Renewable Energy Laboratory have developed a new multi-junction solar cell achieving record efficiency."

sim_result = check_headline_body_consistency(headline, body)
print("Headline-Body Consistency Analysis:")
for k, v in sim_result.items():
    print(f"  {k}: {v}")

## 6. Sentiment Polarity Analysis (VADER)

In [ ]:
from src.sentiment_analyzer import analyze_sentiment

sample_sensational = "SHOCKING CONSPIRACY: Secret alien underground station exposed in catastrophic scandal!"
sent_result = analyze_sentiment(sample_sensational)
print("Sentiment Evaluation:")
for k, v in sent_result.items():
    print(f"  {k}: {v}")

## 7. Named Entity Recognition (spaCy)

In [ ]:
from src.ner_analyzer import extract_entities

sample_ner_text = "Dr. Elena Martinez and Chairman Jerome Powell met at the Federal Reserve in Washington on September 2026 to discuss $500 million clean energy grants."
ner_result = extract_entities(sample_ner_text)
print(f"Engine: {ner_result['engine']}")
print("Entities by Category:")
for label, ents in ner_result['entities_by_type'].items():
    print(f"  {label}: {ents}")

## 8. Explainable AI: LIME Feature Attribution

In [ ]:
from src.explainability import explain_prediction

test_claim = "SHOCKING: Secret Government Alien Base Exposed Inside Mount Everest"
exp_data = explain_prediction(test_claim)
print(f"Explainability Engine: {exp_data['method']}")
print("Top Influential Features:")
for item in exp_data['explanation_list'][:5]:
    print(f"  Word: '{item['word']}', Weight: {item['weight']}, Supports: {item['supports']}")

## 9. Multi-Signal Ensemble Scoring

In [ ]:
from src.fake_score import calculate_fake_score

tfidf_out = {'is_available': True, 'fake_probability': 0.84, 'real_probability': 0.16}
bert_out = {'is_available': True, 'fake_probability': 0.90, 'real_probability': 0.10}
headline_out = {'is_available': True, 'similarity_score': 0.35}

final_verdict = calculate_fake_score(
    tfidf_result=tfidf_out,
    bert_result=bert_out,
    headline_result=headline_out,
    has_headline=True
)

print("=== Final TruthRadar Decision ===")
print(f"Prediction      : {final_verdict['prediction']}")
print(f"Fake Probability: {final_verdict['fake_percentage']}%")
print(f"Real Probability: {final_verdict['real_percentage']}%")
print(f"Confidence Tier : {final_verdict['confidence']}")
print(f"Active Models   : {final_verdict['active_models']}")